# Nemotron Merge Evaluation: tinker_plus_1e5_full_010

Single-merge evaluation kernel.

**Merge:** tinker_plus_1e5_full_010
**Base:** Tinker
**Donor:** 1e-5
**Weight:** 0.1
**Modules:** (all)


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Glob-discover instead of hardcoding — Kaggle's path conventions vary
# (`/kaggle/input/datasets/<owner>/<slug>/` for our newer dataset,
#  `/kaggle/input/<slug>/` for older public ones).
import glob, sys
matches = glob.glob("/kaggle/input/**/blackwell_setup.py", recursive=True)
assert matches, "code dataset not attached (no blackwell_setup.py found under /kaggle/input)"
CODE_ROOT = os.path.dirname(matches[0])
print(f"CODE_ROOT = {CODE_ROOT}")
sys.path.insert(0, CODE_ROOT)

import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Pre-model-load Blackwell setup. The mamba_ssm stub MUST be installed before
# AutoModelForCausalLM.from_pretrained loads modeling_nemotron_h.py.
from blackwell_setup import (  # noqa: E402
    set_cuda_alloc_env,
    set_blackwell_ptxas_env,
    install_mamba_ssm_stub,
)
set_cuda_alloc_env()
set_blackwell_ptxas_env()
install_mamba_ssm_stub()
print("CUDA alloc env + Blackwell ptxas env + mamba_ssm stub ready.")


In [ ]:
# peft + vllm for adapter loading/inference. Some Kaggle images include vllm,
# others do not; install from offline packages to keep this kernel deterministic.
# Internet must be OFF for competition compliance, so `--no-index --find-links`
# is mandatory.
import glob, subprocess, sys
matches = glob.glob("/kaggle/input/**/offline_packages", recursive=True)
assert matches, "offline_packages dir not found under /kaggle/input"
OFFLINE_DIR = matches[0]
print(f"OFFLINE_DIR = {OFFLINE_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir",
    "--no-index", "--find-links", OFFLINE_DIR,
    "peft", "vllm",
])
print("peft + vllm installed from offline packages.")

# Compatibility shim for newer protobuf runtimes where MessageFactory.GetPrototype
# was removed (some Kaggle/papermill-side deps still call it).
from google.protobuf import message_factory as _message_factory
if not hasattr(_message_factory.MessageFactory, "GetPrototype"):
    def _get_prototype(self, descriptor):
        return _message_factory.GetMessageClass(descriptor)
    _message_factory.MessageFactory.GetPrototype = _get_prototype
    print("Applied protobuf MessageFactory.GetPrototype compatibility shim.")


In [ ]:
from pathlib import Path

# Use the kernel-metadata.json-attached mount directly (faster than kagglehub.download).
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
assert BASE_MODEL_PATH.is_dir(), f"base model not mounted at {BASE_MODEL_PATH}"
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")
print("  files:", sorted(p.name for p in BASE_MODEL_PATH.iterdir())[:6], "...")

# val_split_600.csv ships inside the code dataset (see scripts/sync_code_to_kaggle.py).
VAL_SPLIT = Path(CODE_ROOT) / "data" / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Single merge: tinker_plus_1e5_full_010
import sys
from pathlib import Path
import pandas as pd
from nemotron_local_cv import __main__ as cv_cli
import zipfile

SCRIPTS_ROOT = Path(CODE_ROOT) / "scripts"
if SCRIPTS_ROOT.is_dir():
    sys.path.insert(0, str(SCRIPTS_ROOT))
from materialize_merge_experiments import materialize_merge_experiments

# Create single-merge experiment
merge_name = "tinker_plus_1e5_full_010"
base_adapter = r"/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1"
donor_adapter = r"/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/1e-5/1"
donor_weight = 0.1
include_modules = r""

experiment_csv = Path("/kaggle/working/experiment.csv")
pd.DataFrame({
    "name": [merge_name],
    "base_adapter": [base_adapter],
    "donor_adapter": [donor_adapter],
    "donor_weight": [donor_weight],
    "include_modules": [include_modules if include_modules else ""],
}).to_csv(experiment_csv, index=False)

# Materialize
merged_root = Path("/kaggle/working/merged_adapter")
merged_manifest = Path("/kaggle/working/manifest.csv")
print(f"Materializing {merge_name}...")
materialize_merge_experiments(
    experiments_csv=experiment_csv,
    output_root=merged_root,
    manifest_out=merged_manifest,
)

# Evaluate
print(f"Evaluating {merge_name}...")
summary_out = Path("/kaggle/working/eval_summary.csv")
cmd = [
    "nemotron_local_cv",
    "--adapter-manifest",
    str(merged_manifest),
    "--summary-out",
    str(summary_out),
    "--val",
    str(VAL_SPLIT),
    "--model",
    str(BASE_MODEL_PATH),
]
sys.argv = cmd
rc = cv_cli.main()
if rc != 0:
    print(f"ERROR: Evaluation failed with code {rc}")
    exit(1)

# Read result
df = pd.read_csv(summary_out)
accuracy = df["accuracy"].iloc[0]
print(f"✓ {merge_name}: {accuracy:.4f}")

# Package merge as submission.zip
adapter_dir = merged_root / merge_name
REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in REQUIRED if not (adapter_dir / f).is_file()]
if missing:
    print(f"ERROR: Missing files: {missing}")
    exit(1)

submission_zip = Path("/kaggle/working/submission.zip")
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED:
        zf.write(adapter_dir / fname, arcname=fname)

size_mb = submission_zip.stat().st_size / 1024 / 1024
print(f"✓ Packaged {submission_zip} ({size_mb:.2f} MB)")
print(f"✓ Ready for submission!")
